# Language Model Loss Harness



## 0. Setup

Install dependencies once.

In [2]:
%pip install -r requirements.txt

import math
import torch
import torch.nn.functional as F
import tiktoken
from src.model import CausalLanguageModel, count_parameters
from src.losses import IGNORE_INDEX, next_token_loss, horizon_loss, make_targets, chunked_cross_entropy
from src.experiments import print_shapes, print_alignment, decode_token, measure_parameter_tying, benchmark_memory

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
enc = tiktoken.get_encoding("gpt2")
V = enc.n_vocab
PAD_ID = enc.eot_token  # GPT-2 has no dedicated pad token; mask this ID whenever used as padding.
DOC_A = "The cat sat on the mat and then the cat slept."
DOC_B = "A robot learns to predict the next token from context."
a, b = enc.encode(DOC_A), enc.encode(DOC_B)
print(f"Device={DEVICE}; vocabulary={V:,}; pad/eot ID={PAD_ID}")

Device=cpu; vocabulary=50,257; pad/eot ID=50256


## Part 1.1 — Print all important tensor shapes

`B` means batch size, `T` sequence length, `D` model width, and `V` vocabulary size. The vocabulary dimension must be the last dimension of logits.

In [3]:
width = max(len(a), len(b))
tokens = torch.full((2, width), PAD_ID, dtype=torch.long, device=DEVICE)
tokens[0, :len(a)] = torch.tensor(a, device=DEVICE)
tokens[1, :len(b)] = torch.tensor(b, device=DEVICE)
model = CausalLanguageModel(V, d_model=128, n_layer=2, n_head=4, max_len=512).to(DEVICE)
with torch.no_grad():
    hidden = model(tokens)
    logits = model.head1(hidden)
print_shapes(tokens, hidden, logits)
loss, targets = next_token_loss(logits, tokens, PAD_ID)
print(f"Masked next-token loss: {loss.item():.6f}")

Tensor shapes (B=batch, T=sequence positions, D=model width, V=vocabulary):
  tokens             (2, 12)              B sequences x T token IDs
  hidden             (2, 12, 128)         B x T x D contextual vectors
  logits             (2, 12, 50257)       B x T x V scores for vocabulary tokens
  logits[:, :-1]     (2, 11, 50257)       B x (T-1) x V; last position has no next target
  tokens[:, 1:]      (2, 11)              B x (T-1); target is shifted one position forward
  flat logits        (22, 50257)          N x V; N=B*(T-1) prediction rows
  flat targets       (22,)                N target IDs, one per prediction row
Masked next-token loss: 10.853086


## Part 1.2 — Prove the shift using words, not IDs

At each position the input token should be paired with the token immediately after it. GPT-2 BPE tokens may include a leading space; that space is part of the token.

In [4]:
print_alignment(tokens[:1], enc, PAD_ID)


Decoded token alignment (strings, not IDs):
position  input              target t+1         target t+2        
       0  The                 cat                sat              
       1   cat                sat                on               
       2   sat                on                 the              
       3   on                 the                mat              
       4   the                mat                and              
       5   mat                and                then             
       6   and                then               the              
       7   then               the                cat              
       8   the                cat                slept            
       9   cat                slept             .                 


## Part 1.3 — Mask padding

Cross-entropy will happily score padding unless we tell it not to. The reliable check is that the number of contributing targets decreases when the padding mask is applied. The loss itself can go either direction.

In [5]:
shifted_targets = tokens[:, 1:].clone()
shifted_logits = logits[:, :-1, :]
loss_unmasked = F.cross_entropy(shifted_logits.reshape(-1, V), shifted_targets.reshape(-1))
masked_targets = shifted_targets.clone()
masked_targets[masked_targets == PAD_ID] = IGNORE_INDEX
loss_masked = F.cross_entropy(shifted_logits.reshape(-1, V), masked_targets.reshape(-1), ignore_index=IGNORE_INDEX)
print(f"Targets without mask: {shifted_targets.numel()}")
print(f"Targets with mask:    {(masked_targets != IGNORE_INDEX).sum().item()}")
print(f"Loss without mask:    {loss_unmasked.item():.6f}")
print(f"Loss with mask:       {loss_masked.item():.6f}")

Targets without mask: 22
Targets with mask:    21
Loss without mask:    10.841351
Loss with mask:       10.853086


## Part 1.4 — Pack documents and mask the boundary

Packing saves unused space, but without a seam mask the last token of document A is trained to predict the first token of document B. This section compares the mean loss before and after excluding that one target.

In [6]:
packed = torch.tensor([a + b], dtype=torch.long, device=DEVICE)
with torch.no_grad(): packed_hidden = model(packed)
packed_logits = model.head1(packed_hidden[:, :-1])
packed_targets = packed[:, 1:]
per_position = F.cross_entropy(packed_logits.reshape(-1, V), packed_targets.reshape(-1), reduction="none")
seam = len(a) - 1
keep = torch.ones_like(packed_targets, dtype=torch.bool); keep[0, seam] = False
print(f"Seam pair: {decode_token(enc, packed[0,seam], PAD_ID)!r} -> {decode_token(enc, packed[0,seam+1], PAD_ID)!r}")
print(f"Before mask: loss={per_position.mean().item():.6f}, count={per_position.numel()}")
print(f"After mask:  loss={per_position[keep.reshape(-1)].mean().item():.6f}, count={keep.sum().item()}")

Seam pair: '.' -> 'A'
Before mask: loss=10.842791, count=22
After mask:  loss=10.845577, count=21


## Part 1.5 — Perplexity sanity check

Perplexity is `exp(loss)`. For near-uniform predictions, cross-entropy is near `ln(V)` and perplexity is near the vocabulary size. Do not expect exact equality from random initialization.

In [7]:
fresh = CausalLanguageModel(V, d_model=128, n_layer=2, n_head=4, max_len=512).to(DEVICE)
with torch.no_grad():
    h = fresh(tokens); lg = fresh.head1(h[:, :-1]); tg = make_targets(tokens, PAD_ID, horizon=1)
    untrained_loss = F.cross_entropy(lg.reshape(-1,V), tg.reshape(-1), ignore_index=IGNORE_INDEX)
ppl = math.exp(min(untrained_loss.item(), 30))
print(f"V={V:,}; ln(V)={math.log(V):.4f}; loss={untrained_loss.item():.4f}; perplexity={ppl:,.1f}; perplexity/V={ppl/V:.3f}")

V=50,257; ln(V)=10.8249; loss=10.8921; perplexity=53,751.7; perplexity/V=1.070


## Part 1.6 — Tied versus untied output head

With tying, the output head shares the input embedding matrix. The expected reduction in unique parameters is exactly `V * D`.

In [8]:
measure_parameter_tying(CausalLanguageModel, V, d_model=128, device=DEVICE)

Untied parameters: 13,328,128
Tied parameters:   6,895,232
Saved:             6,432,896 (expected V*D=6,432,896)
Parameter-count check: PASS


(13328128, 6895232)

## Part 1.7 — Ordinary versus chunked cross-entropy

Ordinary loss creates logits for all positions at once. Chunked loss handles fewer positions at a time. Their loss values should match. On GPU, the code measures peak allocated memory; on CPU, it reports analytic logits tensor sizes because process RSS can be misleading.

In [9]:
mem_model = CausalLanguageModel(V, d_model=64, n_layer=1, n_head=4, max_len=128).to(DEVICE)
sample = torch.randint(0, V-1, (2,128), device=DEVICE)
with torch.no_grad(): mem_hidden = mem_model(sample)[:, :-1].contiguous()
mem_targets = make_targets(sample, PAD_ID, horizon=1)
benchmark_memory(mem_hidden, mem_model.head1, mem_targets, chunk_size=16, device=DEVICE)

CPU forward-only run: analytic logits-memory estimate (RSS is allocator-dependent):
  ordinary logits tensor: 48.70 MiB
  chunked logits tensor:  6.13 MiB
  analytic ratio:         7.94x
  losses: ordinary=10.855759, chunked=10.855759, abs diff=0.000e+00


{'ordinary_loss': 10.855758666992188,
 'chunked_loss': 10.855758666992188,
 'ordinary_logits_mib': 48.69567108154297,
 'chunked_logits_mib': 6.1348876953125}

## Part 2 — Add a head for token `t+2`

The first head predicts the next token. The second predicts the token two positions ahead. Print both losses separately and their sum. **Do not assume the second loss must always be higher**: that is not a mathematical invariant, especially on a tiny repeated training set.

In [10]:
dual = CausalLanguageModel(V, d_model=128, n_layer=2, n_head=4, max_len=512, two_heads=True).to(DEVICE)
optimizer = torch.optim.Adam(dual.parameters(), lr=3e-4)
train_tokens = torch.tensor([a+b], dtype=torch.long, device=DEVICE).repeat(8,1)
for step in range(81):
    h = dual(train_tokens)
    loss1, _ = horizon_loss(dual.head1, h, train_tokens, PAD_ID, horizon=1)
    loss2, _ = horizon_loss(dual.head2, h, train_tokens, PAD_ID, horizon=2)
    total = loss1 + loss2
    if step % 10 == 0:
        print(f"step={step:3d} | loss t+1={loss1.item():.4f} | loss t+2={loss2.item():.4f} | sum={total.item():.4f}")
    if step < 80:
        optimizer.zero_grad(set_to_none=True); total.backward(); optimizer.step()
print(f"One-head parameters={count_parameters(model):,}; two-head parameters={count_parameters(dual):,}")
print("Interpretation: both losses should generally fall on this tiny repeated training set. Evaluate on held-out text before making claims about which horizon is harder.")

step=  0 | loss t+1=10.7201 | loss t+2=10.8964 | sum=21.6164
step= 10 | loss t+1=9.5277 | loss t+2=9.5873 | sum=19.1150
step= 20 | loss t+1=8.6713 | loss t+2=8.6819 | sum=17.3533
step= 30 | loss t+1=7.9632 | loss t+2=7.9850 | sum=15.9481
step= 40 | loss t+1=7.3085 | loss t+2=7.3347 | sum=14.6431
step= 50 | loss t+1=6.6275 | loss t+2=6.6604 | sum=13.2879
step= 60 | loss t+1=5.9083 | loss t+2=5.9321 | sum=11.8404
step= 70 | loss t+1=5.1470 | loss t+2=5.1674 | sum=10.3144
step= 80 | loss t+1=4.3652 | loss t+2=4.3827 | sum=8.7479
One-head parameters=13,328,128; two-head parameters=19,761,024
Interpretation: both losses should generally fall on this tiny repeated training set. Evaluate on held-out text before making claims about which horizon is harder.


## Submission checklist

- [ ] Keep the executed output from every section.
- [ ] Explain each result in your own words.
- [ ] If you change runtime or model settings, rerun all cells.
- [ ] Report CPU memory estimates honestly; use GPU peak allocation for measured peak memory.
- [ ] State that Part 2 uses a tiny repeated text example and does not establish generalization.